# IRED Latent Diffusion — Full Training Notebook

End-to-end pipeline for the **gensis.md §8** experiment:

1. **Milestone 1** — train the AttentionPool on top of a frozen flan-t5 so that `Decoder(Pool(Encoder(A))) ≈ A`.
2. **Milestone 2** — freeze the AE, train the IRED energy network on `(z_q, z_a)` pairs.
3. **Milestone 3** — sweep `inner_steps` and plot accuracy vs. test-time compute.

Defaults match the *Practical defaults that survived first-run debugging* section of gensis.md (full answer mode, `K=32`, cosine β schedule, `weight_decay=0.01` on the EBM, SDPA math kernel).

Runs end-to-end on a single GPU. On CPU, lower `STEPS_AE` / `STEPS_EBM` for a smoke test only — real convergence needs a GPU.

## Colab / fresh-environment setup

No-op when run locally with the repo already installed. On Colab (or any clean env where the `ired` package isn't importable), this cell:

1. Detects whether the `ired` package is available.
2. If not, gets the repo into the current working directory. Three options — uncomment the one you want:
    - **clone** from a remote (fastest if the code is on GitHub),
    - **mount Google Drive** and `cd` into a copy you've already pushed there, or
    - manually **upload** the `ired/` directory + `pyproject.toml` via Colab's file panel.
3. `pip`-installs the runtime deps (Colab doesn't ship `uv`).
4. Adds the repo root to `sys.path` so `from ired.* import ...` resolves.

Recommended GPU: Colab Free T4 fits `flan-t5-base` at `batch_size=16, max_a_length=256`. Sessions disconnect after ~90 min idle — checkpoints land in `checkpoints/ae/` and `checkpoints/ebm/`; you can mount Drive and point the checkpoint dirs there to survive disconnects.

In [ ]:
import sys, os, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
HAS_IRED = importlib.util.find_spec("ired") is not None

if IN_COLAB and not HAS_IRED:
    # ---- 1) GET THE CODE INTO CWD --------------------------------------
    # Pick ONE of the three options below.

    # Option A: clone from GitHub. Replace the URL with your fork.
    # REPO_URL = "https://github.com/<user>/ired-reasoning.git"
    # if not os.path.isdir("ired-reasoning"):
    #     subprocess.run(["git", "clone", REPO_URL], check=True)
    # os.chdir("ired-reasoning")

    # Option B: mount Drive and use a copy you've pushed there.
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/ired-reasoning")  # adjust path

    # Option C: upload `ired/` + `pyproject.toml` via the Files sidebar,
    # then leave cwd as /content (the default).

    if not os.path.isdir("ired"):
        raise RuntimeError(
            "Colab detected but no 'ired/' package found in cwd. "
            "Uncomment one of the options above (clone / Drive / upload) and re-run."
        )

    # ---- 2) INSTALL DEPS (Colab usually has torch + transformers already) ----
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch>=2.5.0", "transformers>=4.45.0", "datasets>=2.20.0",
         "tqdm", "sentencepiece", "protobuf>=4.25.0", "accelerate>=0.34.0"],
        check=True,
    )

    # ---- 3) MAKE `ired` IMPORTABLE -----------------------------------------
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    HAS_IRED = importlib.util.find_spec("ired") is not None

if HAS_IRED:
    print(f"ired package importable from: {os.getcwd()}")
else:
    print("ired package NOT importable — fix the setup options above before continuing.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RuntimeError: Colab detected but no 'ired/' package found in cwd. Uncomment one of the options above (clone / Drive / upload) and re-run.

## Configuration

Edit this cell to override hyperparameters. Everything downstream reads from `CONFIG`.

In [ ]:
import os, time, math, torch

CONFIG = {
    # model
    "model_name":    "google/flan-t5-base",
    "k":             32,
    "pool_layers":   2,
    "pool_heads":    8,
    "ebm_layers":    4,
    "ebm_heads":     8,
    "ebm_ff_mult":   4,

    # data
    "answer_mode":   "full",   # gensis §8: 'full' >> 'final' for K=32 latent capacity
    "max_q_length":  256,
    "max_a_length":  256,

    # diffusion
    "timesteps":     10,
    "inner_steps":   5,
    "beta_schedule": "cosine",
    "opt_step_size": 1.0,
    "x_start_clamp": 5.0,
    "envelope_sf":   None,
    "nce_scale":     1.0,
    "supervise_nce": True,

    # optional decoder-CE auxiliary (see gensis §7.4)
    "decoder_aux_weight": 0.0,
    "decoder_aux_t_max":  2,

    # training
    "batch_size":    16,
    "steps_ae":      2000,
    "steps_ebm":     5000,
    "lr":            3e-4,
    "weight_decay":  0.01,    # gensis §8: load-bearing for the EBM head
    "log_every":     50,
    "eval_every":    500,

    # IO
    "ae_dir":        "checkpoints/ae",
    "ebm_dir":       "checkpoints/ebm",
    "device":        "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":   2,
    "seed":          0,
}

torch.manual_seed(CONFIG["seed"])
os.makedirs(CONFIG["ae_dir"],  exist_ok=True)
os.makedirs(CONFIG["ebm_dir"], exist_ok=True)
print(f"device: {CONFIG['device']}")

In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

from ired.autoencoder import FrozenT5Autoencoder
from ired.data import GSM8KDataset, collate, extract_final_answer
from ired.diffusion import GaussianLatentDiffusion
from ired.energy_net import DiffusionWrapper, EnergyTransformer
from ired.train_autoencoder import eval_reconstruction
from ired.train_diffusion import eval_accuracy

## Data

Builds the GSM8K train/test loaders once and reuses them for both milestones.

In [ ]:
train_ds = GSM8KDataset("train", answer_mode=CONFIG["answer_mode"])
test_ds  = GSM8KDataset("test",  answer_mode=CONFIG["answer_mode"])

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=collate, num_workers=CONFIG["num_workers"], drop_last=True,
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    collate_fn=collate, num_workers=CONFIG["num_workers"],
)
print(f"train: {len(train_ds)}  test: {len(test_ds)}  answer_mode={CONFIG['answer_mode']}")

## Milestone 1 — Train the AttentionPool

Frozen T5 + a learned `K=32` attention pool. Trains by reconstruction CE: `Decoder(Pool(Encoder(A))) → A`.

**Decision rule (gensis §8):** ≥ 95% exact match in `final` mode, ≥ 90% final-answer extraction in `full` mode. If you don't clear this, the EBM cannot recover what Milestone 1 loses — retrain with more steps or a bigger pool.

In [ ]:
ae = FrozenT5Autoencoder(
    model_name=CONFIG["model_name"],
    k=CONFIG["k"],
    pool_layers=CONFIG["pool_layers"],
    pool_heads=CONFIG["pool_heads"],
).to(CONFIG["device"])
ae.train()

pool_params = ae.trainable_parameters()
print(f"trainable pool params: {sum(p.numel() for p in pool_params):,}")
print(f"d_model: {ae.d_model}")

In [ ]:
opt = AdamW(pool_params, lr=CONFIG["lr"])
step = 0
train_iter = iter(train_loader)
t0 = time.time()

while step < CONFIG["steps_ae"]:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    z = ae.encode_to_latents(batch["answer"], CONFIG["device"], max_length=CONFIG["max_a_length"])
    loss = ae.decode_loss(z, batch["answer"], CONFIG["device"], max_length=CONFIG["max_a_length"])

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(pool_params, 1.0)
    opt.step()

    step += 1
    if step % CONFIG["log_every"] == 0:
        print(f"[ae] step {step:6d} | loss {loss.item():.4f} | {time.time()-t0:.1f}s")

    if step % CONFIG["eval_every"] == 0 or step == CONFIG["steps_ae"]:
        acc, avg = eval_reconstruction(
            ae, test_loader, CONFIG["device"], CONFIG["max_a_length"], n_batches=10,
        )
        print(f"  [eval] recon acc {acc:.3f} | loss {avg:.4f}")
        ckpt = {"pool": ae.state_dict_pool(), "config": CONFIG, "step": step, "eval_acc": acc}
        torch.save(ckpt, os.path.join(CONFIG["ae_dir"], f"pool_step{step}.pt"))
        torch.save(ckpt, os.path.join(CONFIG["ae_dir"], "pool_latest.pt"))

print(f"\nMilestone 1 done. Last recon acc: {acc:.3f}")

## Milestone 2 — Train the IRED energy network

Freeze everything in the AE (T5 + pool). Train only the `EnergyTransformer`.

**Per-step monitoring:**
- `mse, nce` — denoising MSE + NCE energy contrast loss.
- `e_real, e_fake` — absolute energies of clean vs. mined-fake samples. Watch the **ratio**, not the absolutes.
- `eps_scale = ||ε̂|| / ||noise||` — should stay near 1.0. Drift signals head-magnitude inflation that breaks DDPM regardless of MSE.

**Per-eval monitoring:**
- `ae_recon_acc` — should stay at Milestone 1's level; if it drops, the load path broke.
- `ebm_acc(inner=N)` vs `ebm_acc(inner=0)` — isolates whether `opt_step` is helping.
- `mse_z, std_zs, corr_z` — direction (`corr_z`) vs. magnitude (`std_zs`) of where `z_sampled` lands relative to `z_a`.

In [ ]:
ae.eval()
for p in ae.parameters():
    p.requires_grad_(False)
print("AE frozen.")

In [ ]:
d_model = ae.d_model
ebm = EnergyTransformer(
    d_model=d_model,
    k=CONFIG["k"],
    n_layers=CONFIG["ebm_layers"],
    n_heads=CONFIG["ebm_heads"],
    dim_ff_mult=CONFIG["ebm_ff_mult"],
).to(CONFIG["device"])
wrapper = DiffusionWrapper(ebm).to(CONFIG["device"])

diffusion = GaussianLatentDiffusion(
    model=wrapper,
    latent_shape=(CONFIG["k"], d_model),
    timesteps=CONFIG["timesteps"],
    beta_schedule=CONFIG["beta_schedule"],
    opt_step_size=CONFIG["opt_step_size"],
    loss_scale=CONFIG["nce_scale"],
    supervise_energy_landscape=CONFIG["supervise_nce"],
    x_start_clamp=CONFIG["x_start_clamp"],
    envelope_sf=CONFIG["envelope_sf"],
    decoder_aux_weight=CONFIG["decoder_aux_weight"],
    decoder_aux_t_max=CONFIG["decoder_aux_t_max"],
).to(CONFIG["device"])
diffusion.train()

if CONFIG["decoder_aux_weight"] > 0:
    diffusion.set_decoder_loss_fn(
        lambda x, txt: ae.decode_loss(x, txt, CONFIG["device"], max_length=CONFIG["max_a_length"])
    )

print(f"ebm params: {sum(p.numel() for p in ebm.parameters()):,}  (d_model={d_model}, layers={CONFIG['ebm_layers']})")

In [ ]:
opt = AdamW(ebm.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
step = 0
train_iter = iter(train_loader)
t0 = time.time()

while step < CONFIG["steps_ebm"]:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    with torch.no_grad():
        z_q = ae.encode_to_latents(batch["question"], CONFIG["device"], max_length=CONFIG["max_q_length"])
        z_a = ae.encode_to_latents(batch["answer"],   CONFIG["device"], max_length=CONFIG["max_a_length"])

    loss, stats = diffusion(
        z_q, z_a,
        gold_texts=batch["answer"] if CONFIG["decoder_aux_weight"] > 0 else None,
    )
    if not torch.isfinite(loss):
        raise RuntimeError(f"step {step}: non-finite loss {loss.item()} (stats={stats})")

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ebm.parameters(), 1.0)
    opt.step()

    step += 1
    if step % CONFIG["log_every"] == 0:
        extras = " | ".join(f"{k} {v:.4f}" for k, v in stats.items())
        print(f"[ebm] step {step:6d} | loss {loss.item():.4f} | {extras} | {time.time()-t0:.1f}s")

    if step % CONFIG["eval_every"] == 0 or step == CONFIG["steps_ebm"]:
        ev = eval_accuracy(
            ae, diffusion, test_loader, CONFIG["device"],
            CONFIG["max_q_length"], CONFIG["max_a_length"],
            CONFIG["inner_steps"], n_batches=5,
        )
        print(
            f"  [eval n={ev['n']}] "
            f"ebm_acc(inner={CONFIG['inner_steps']})={ev['acc']:.3f}  "
            f"ebm_acc(inner=0)={ev['acc_inner0']:.3f}  "
            f"ae_recon_acc={ev['ae_acc']:.3f}"
        )
        print(
            f"  [latent] mse_z={ev['mse_z']:.3f}  mse_z(inner=0)={ev['mse_z_inner0']:.3f}  "
            f"std_za={ev['std_za']:.3f}  std_zs={ev['std_zs']:.3f}  corr_z={ev['corr_z']:+.3f}"
        )
        ckpt = {
            "ebm": ebm.state_dict(), "config": CONFIG, "step": step,
            "eval_acc": ev["acc"], "ae_recon_acc": ev["ae_acc"], "mse_z": ev["mse_z"],
        }
        torch.save(ckpt, os.path.join(CONFIG["ebm_dir"], f"ebm_step{step}.pt"))
        torch.save(ckpt, os.path.join(CONFIG["ebm_dir"], "ebm_latest.pt"))

print(f"\nMilestone 2 done. Last ebm_acc: {ev['acc']:.3f}")

## Milestone 3 — Test-time compute sweep

Sweep `inner_steps ∈ {0, 1, 2, 5, 10}` on the test set and report exact-final-answer accuracy.

**Decision rule (gensis §8):** a steeper accuracy-vs-compute curve than AR-CoT at matched FLOPs validates the thesis. A flat or decreasing curve is an informative negative about smoothness of reasoning in latent space.

In [ ]:
diffusion.eval()
sweep = [0, 1, 2, 5, 10]
n_eval_batches = 20
results = []

print(f"{'inner':>6}  {'acc':>6}  {'n':>5}  {'ebm_passes':>11}  {'time_s':>7}")
for n_inner in sweep:
    t = time.time()
    correct = total = 0
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if i >= n_eval_batches:
                break
            z_q = ae.encode_to_latents(
                batch["question"], CONFIG["device"], max_length=CONFIG["max_q_length"],
            )
            z = diffusion.sample(z_q, inner_steps=n_inner)
            preds = ae.decode(z, max_length=CONFIG["max_a_length"])
            for p, a in zip(preds, batch["answer"]):
                total += 1
                if extract_final_answer(p) == extract_final_answer(a):
                    correct += 1
    acc = correct / max(total, 1)
    ebm_passes = CONFIG["timesteps"] * (1 + n_inner)
    dt = time.time() - t
    print(f"{n_inner:>6}  {acc:>.3f}  {total:>5}  {ebm_passes:>11}  {dt:>7.1f}")
    results.append({"inner_steps": n_inner, "acc": acc, "n": total, "ebm_passes": ebm_passes, "time_s": dt})

### Optional: plot the curve

Requires `matplotlib`. Skip if running headless.

In [ ]:
try:
    import matplotlib.pyplot as plt
    xs = [r["ebm_passes"] for r in results]
    ys = [r["acc"] for r in results]
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("EBM forward+backward passes per sample")
    plt.ylabel("final-answer accuracy")
    plt.title("Test-time compute curve (IRED in latent space)")
    plt.grid(True, alpha=0.3)
    plt.show()
except ImportError:
    print("matplotlib not installed; skip plot.")

## What the result tells you

- **Milestone 1 < ~90% extraction** → the pool is the bottleneck; the EBM can't recover what's lost here. Train longer / bigger pool / fewer compression (smaller `K` doesn't help — *larger* would).
- **Milestone 2 `eps_scale` drifts away from 1.0** → head-magnitude inflation. Increase `weight_decay`, lower `lr`, or check for any change to the energy parameterization.
- **Milestone 2 `corr_z ≈ 0`** → EBM hasn't learned a useful denoising direction yet. Train longer or revisit the contrast strength.
- **Milestone 2 `std_zs >> std_za`** → reverse process inflates magnitude. Usually a downstream symptom of `eps_scale` drift.
- **Milestone 3 curve flat / decreasing in `inner_steps`** → either the energy landscape isn't calibrated (NCE collapsed) or `opt_step` is over-stepping. Try lower `opt_step_size` or check `e_real/e_fake` history.
- **Curve monotonically rising with `inner_steps`** → IRED's thesis is validating. Compare against AR-CoT at matched FLOPs.